## After wrangling and exploring the data in previous notebooks, this notebook will explore feature engineering and then forecasting the total electricity load using regression, time series forecasting, and ensemble methods.

In [ ]:
#import standard packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

#import preprocessing/modeling/error metric packages
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor

from energy_forecast.notebook_setup import DATA


In [ ]:
combined_avg = pd.read_csv(DATA / 'combined_avg.csv', index_col='time', parse_dates=True)

In [ ]:
combined_avg.head(2)

In [ ]:
combined_avg.info()

In [ ]:
#function to make train-test split for time-indexed data
def ts_train_test(data, target_col_name = 'total load actual', test_size=0.15, stdzd=False, cols_to_scale=None):
    df = data.copy()
    test_index = int(len(df)*(1-test_size)) #get index where test set begins
        
    X_train = df.drop([target_col_name], axis = 1).iloc[:test_index]
    y_train = df[target_col_name].iloc[:test_index]
    X_test = df.drop([target_col_name], axis = 1).iloc[test_index:]
    y_test = df[target_col_name].iloc[test_index:]
    
    # StandardScaler fit seperately on training and test sets
    if stdzd == True:
        scaler = StandardScaler()
        X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
        X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])
    
    return X_train, X_test, y_train, y_test        

In [ ]:
#Let's engineer some categorical features for use in a regression model like weekend/weekday, winter/summer/spring-fall
df_features = combined_avg.iloc[:,16:].drop(['price day ahead', 'price actual','total load forecast'], axis=1)
features_min = df_features.copy()
df_features.head()

In [ ]:
#function to calculate basic season label based on month
def season_determination(month):
    if month in [6,7,8,9]: #June-Sept = summer (highest need for cooling in Spain)
        return "summer"
    elif month in [1,2,12]: #Dec, Jan, Feb = winter (highest need for heating)
        return "winter"
    else:
        return "spring/fall" #all other months are spring or fall (similar lower needs for heating/cooling)

In [ ]:
day_of_week = {0:'Monday', 1:'Tuesday', 2:'Wednesday', 3: 'Thursday', 4: 'Friday', 5:'Saturday', 6:'Sunday'}
df_features['hour'] = df_features.index.hour
df_features['weekday'] = df_features.index.weekday.map(day_of_week)
df_features['month'] = df_features.index.month #have to create month column because cannot apply() on datetimeindex
df_features['season'] = df_features.month.apply(season_determination)
df_features['nonwork-work_day'] = np.where(df_features.index.weekday > 5, 0, 1)
display(df_features.shape, df_features.head());

In [ ]:
df_features.drop("month", axis=1, inplace=True)

Now that I've used the weekday and month to create the type_of_day and season columns, 
I need to drop the month column because the test set may not contain all months. Now time to one-hot encode the new categorical variables.

In [ ]:
#let's split data and then encode the new catergorical variables
X_train, X_test, y_train, y_test = ts_train_test(data = df_features, stdzd=True, 
                                                 cols_to_scale=['temp','pressure','humidity',
                                                                'wind_speed','rain_1h','snow_3h','clouds_all'])
cat_cols = ['hour','weekday','season']
X_train_cat = X_train[['hour','weekday','season']]
X_test_cat = X_test[['hour','weekday','season']]

encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

encoder.fit(X_train_cat)
X_train.drop(columns = cat_cols, inplace=True)
X_train_cat = pd.DataFrame(encoder.transform(X_train_cat), index=X_train.index, 
                           columns = encoder.get_feature_names_out()).astype(int)
X_train = X_train.join(X_train_cat, how ='outer')
#X_train.head()
encoder.fit(X_test_cat)
X_test.drop(columns = cat_cols, inplace=True)
X_test_cat = pd.DataFrame(encoder.transform(X_test_cat), index = X_test.index, 
                           columns = encoder.get_feature_names_out()).astype(int)
X_test = X_test.join(X_test_cat, how ='outer')

display(X_train.head(3), X_test.head(3), X_train.columns)

So now X is split into train and test sets, with the first 85% of the data going to training to try to predict approximately the last 7 months out of the four years of data.

# Modeling

First I have written a couple helper functions for calculating error metrics and for creating a TS plot comparing predicted vs actual load for the various forecasting models I will try out.

### Helper functions

In [ ]:
#function for calculating and presenting error metrics, and storing in a dict for comparison at the end
error_dict = {} #dict to hold model name and error metrics for various models that are investigated

def error_metrics(y_true, y_pred, model_name = None):
    
    #function will print RMSE, R2, MAE, MAPE. Assumes y_pred is np array
    
    RMSE = np.sqrt(mean_squared_error(y_true, y_pred))
    R2 = r2_score(y_true, y_pred)
    MAE = mean_absolute_error(y_true, y_pred)
    MAPE = (np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

    print('\nError metrics for model: {}'.format(model_name))
    print("RMSE: %.2f" % RMSE)
    print('Variance/R^2: %.2f' % R2)
    print('MAE: %.2f' % MAE)
    print('Mean Absolute Percentage Error: %.2f %%' % MAPE)
    
    key = ['Model Name','RMSE', 'R2', 'MAE', 'MAPE']
    value = [model_name, RMSE, R2, MAE, MAPE]
    pair = list(zip(key, value))
    '''
    for error in pair:
        error_dict[error[0]]= [error[1]]
    '''
    for error in pair:
        if error[0] in error_dict:
            error_dict[error[0]].append(error[1])
        else:
            error_dict[error[0]]= [error[1]]
            

In [ ]:
#function for plotting time series of predicted vs true values
def plot_ts_pred_true(y_pred, y_true, model_name=None):
    fig, ax = plt.subplots(figsize =(15,10))
    ax.plot(y_true.index, y_pred, linestyle='-', linewidth=1, label = 'Model Forecasted Total Load', color = 'blue',alpha = 0.4)
    y_true.plot(linestyle='-', linewidth=1, label = 'Actual Total Load', color = 'red',alpha = 0.4)

    plt.ylabel('Load/Demand (MW)')
    plt.xlabel("Time")
    plt.title("Observed vs model-predicted total load (MWH) using {}".format(model_name))
    plt.legend()
    plt.show()

First I will try out a simple linear regression using the weather and categorical variables:
### Linear Regression

In [ ]:
#instantiate and fit linear regression model
linreg = LinearRegression()
linreg.fit(X_train, y_train)

In [ ]:
#calculate error metrics for train set
error_metrics(linreg.predict(X_train), y_train, model_name = 'simple linear regression (train)')

In [ ]:
#calculate error metrics for test set
error_metrics(linreg.predict(X_test), y_test, model_name = 'simple linear regression (test)')

In [ ]:
plot_ts_pred_true(y_pred = linreg.predict(X_test), y_true = y_test, model_name = "simple linear regression")

In [ ]:
#let's try using a df with reduced feature space, by eliminating some possibly extraneous variables:

X_train_red = X_train.drop(['hour_1', 'hour_2', 'hour_3', 'hour_4', 'hour_5', 'hour_6', 'hour_7', 'hour_8', 'hour_9',
       'hour_10', 'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18', 
       'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'weekday_Monday', 'weekday_Saturday', 'weekday_Sunday', 
       'weekday_Thursday', 'weekday_Tuesday','weekday_Wednesday',], axis=1)
X_test_red = X_test.drop(['hour_1', 'hour_2', 'hour_3', 'hour_4', 'hour_5', 'hour_6', 'hour_7', 'hour_8', 'hour_9',
       'hour_10', 'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18', 
       'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'weekday_Monday', 'weekday_Saturday', 'weekday_Sunday', 
       'weekday_Thursday', 'weekday_Tuesday','weekday_Wednesday',], axis=1)

### Linear regression on reduced feature space

In [ ]:
#instantiate and fit linear regression model for reduced feature space train set
linreg_red = LinearRegression()
linreg_red.fit(X_train_red, y_train)

In [ ]:
#calculate error metrics for linear regr with reduced feature train set
error_metrics(y_train, linreg_red.predict(X_train_red), model_name = 'simple linear regression on reduced features (train)')

We can see that the simple linear regression model using the larger set of features predicts the weekly seasonality fairly well but is often over or underpredicting the max daily load. The model using the reduced feature space performs worse. I will also check out a baseline model that just predicts the same as the value for the same date/time from the previous year:
### Simple Year-over-Year Baseline 

In [ ]:
# errors metrics for a baseline forecast (that simply repeats the values from the previous year)
#error_metrics(y_true, y_pred, model_name = None)
error_metrics(y_test, df_features.loc[X_test.index.shift(-8760, freq='H'), 'total load actual'],
              model_name='Baseline forecast (repeat of previous year) (test)')


In [ ]:
# plot y_pred and y-true for baseline year over year
plot_ts_pred_true(y_pred = df_features.loc[X_test.index.shift(-8760, freq='H'), 'total load actual'], y_true = y_test, 
                  model_name = "baseline forecast that repeats the value from the previous year")

Visually we can can see that the baseline model that just uses the value from the same time from the previous year does a decent job, but misses is off pretty badly sometimes. Which makes sense, because on average things are similar year to year but the peak demand will definitely be much higher or much lower on certain days. Let's try out a couple more regression models before trying some time series forecasting:

### Random Forest

In [ ]:
#set up a parameter grid of different values for GridSearchCV of number of trees and max depth
n_est = [int(n) for n in np.logspace(start=1, stop=2.5, num=12)]
max_depth = list(range(1,6))

param_grid = {
        'n_estimators': n_est,
        'max_depth': max_depth
}
param_grid

In [ ]:
#create instance of base model
rfreg = RandomForestRegressor()

#create TS splits for cross val (instead of random splits, this uses progressively larger sets starting from beginning)
tss = TimeSeriesSplit(n_splits=5)

#create instance of RandomSearchCV
rf_cv = RandomizedSearchCV(rfreg, param_distributions=param_grid, cv=tss, random_state=47)

# Fit the random search model using reduced feature space X set
rf_cv.fit(X_train_red, y_train)

rf_cv.best_params_


The R^2 score:

In [ ]:
rf_cv.score(X_train_red, y_train)

In [ ]:
rf_cv.score(X_test_red, y_test)

In [ ]:
#RF error metrics
error_metrics(y_train, rf_cv.predict(X_train_red),  
                  model_name = 'Random Forest Regression on reduced feature space tuned with Random Search CV (train)')

Let's also try Random Forest on the full feature space dataset:

In [ ]:
#create instance of base model
rfreg = RandomForestRegressor()

#create TS splits for cross val (instead of random splits, this uses progressively larger sets starting from beginning)
tss = TimeSeriesSplit(n_splits=5)

#create instance of RandomSearchCV
rf_cv_full = RandomizedSearchCV(rfreg, param_distributions=param_grid, cv=tss, random_state=47)

# Fit the random search model using reduced feature space X set
rf_cv_full.fit(X_train, y_train)

rf_cv_full.best_params_

In [ ]:
rf_cv_full.score(X_train, y_train)

In [ ]:
rf_cv_full.score(X_test, y_test)

In [ ]:
error_metrics(y_train, rf_cv_full.predict(X_train), 
                  model_name = 'Random Forest Regression tuned with Random Search CV (train)')

In [ ]:
error_metrics(y_test, rf_cv_full.predict(X_test), 
                  model_name = 'Random Forest Regression tuned with Random Search CV (test)')

In [ ]:
plot_ts_pred_true(y_pred = rf_cv_full.predict(X_test), y_true = y_test, model_name = "Random Forest Regression")

So Random Forest performs slightly worse that linear regression. It seems to consistently underpredict the true values. Let's also try KNeighbors Regression:
###  KNN

In [ ]:
#set up a parameter grid for KNeighbors Random CV search:

n_neigh = [int(n) for n in np.logspace(start=0.5, stop=1.8, num=10)]
weights = ['uniform', 'distance']
leaf_size = [int(n) for n in np.logspace(start=0.5, stop=1.9, num=6)]

kn_param_grid = {
        'n_neighbors': n_neigh,
        'weights': weights,
        'leaf_size':leaf_size
}
kn_param_grid

In [ ]:
#create instance of base model
kn = KNeighborsRegressor()

#create TS splits for cross val (instead of random splits, this uses progressively larger sets starting from beginning)
tss = TimeSeriesSplit(n_splits=5)

#create instance of RandomSearchCV
kn_cv = RandomizedSearchCV(kn, param_distributions = kn_param_grid, cv=tss, random_state=47)

# Fit the random search model using reduced feature space X set
kn_cv.fit(X_train, y_train)

kn_cv.best_params_


In [ ]:
kn_cv.score(X_train, y_train)

In [ ]:
kn_cv.score(X_test, y_test)

In [ ]:
error_metrics(y_test, kn_cv.predict(X_test), 
                  model_name = 'KNN tuned with Random Search CV (test)')

In [ ]:
plot_ts_pred_true(y_pred = kn_cv.predict(X_test), y_true = y_test, model_name = "Tuned KNN on test set")

As expected, this model wildly overfits the training data and performs similarly to Random Forest and linear regression on the test data. Now to try some time series forecasting.

# Time Series Forecasting

In [ ]:
# sktime must be available in *this* notebook kernel (same Python Jupyter uses).
# If you see ModuleNotFoundError: either run the line below once, or in a terminal:
#   .venv/bin/python -m pip install sktime
# and select the .venv kernel in VS Code / Jupyter.
%pip install sktime -q

import sktime
from statsmodels.graphics import tsaplots
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

In [ ]:
# a simple time series DF for use in models without the exogenous variables
ts_load = pd.DataFrame(combined_avg["total load actual"], columns=['total load actual'])
ts_load.head()

With the univariate TS dataframe, I will use the statsmodels Augmented Dickey-Fuller test to test for stationarity. The null hypothesis is that the series is not stationary (no trend, constant variance over time, constant autocorrelation structure over time) and the alternative hypothesis is that the time series is stationary. If the p-value is less than 0.05, we reject the null hypothesis and conclude the series is stationary. 

In [ ]:
#let's use the simple ts df to test for stationarity, as many ts models assume stationarity
adf_result = adfuller(ts_load)
#print test statisitc
print("t-stat", adf_result[0])
# Print p-value
print("p-value", adf_result[1])
# Print number of lags used
print("#lags used", adf_result[2])
# Print critical values
print("critical values", adf_result[4]) 

With a p-value of zero, we can say that the data are stationary. Even though we know the data have seasonality, perhaps the p-value is so low because there is very little trend. But let's have a look at the decomposition of the trend, seasonality, and residuals. Let's start with the largest seasonal component, the yearly seasonality:

In [ ]:
#seasonal decomposition
decomp = sm.tsa.seasonal_decompose(ts_load['total load actual'], period=24*365) #the yearly seasonal component
#the main seasonal component is the shift from winter to spring to summer to fall and back to winter an so on...
# so the period is 24*365

_ = decomp.plot() #plot the seasonal components

So the seasonal decomposition actually does show a trend, but it is so small as to almost be negligible (an increase of about 900 MWH out of about 30000 over 4 years). The seasonal component plot clearly shows the peaks during summer and winter and relative lows in spring and fall.

In [ ]:
# Differencing the data once
#decomp_diff1 = sm.tsa.seasonal_decompose(ts_load['total load actual'].diff().dropna(), period=24*365) 
#_ = decomp_diff1.plot()

In [ ]:
#Decomposition on the data with 24-hour differencing, 24 being the period of the shortest seasonality in the dataset
#decomp_diff24 = sm.tsa.seasonal_decompose(ts_load['total load actual'].diff(24).dropna(), period=24*365) 
#_ = decomp_diff24.plot()

Now I will use the FourierFeatures class from sktime to create Fourier terms for the multiple seasonalities so that we can use the SARIMAX forecasting model (which only handles a single seasonality). SARIMAX tends to handle short seasonality better than long seasonality, so I can either create Fourier terms for all the seasonalities, or create them only for the yearly/seasonal and weekly seasonalities and let the model handle the daily seasonality.

In [ ]:
from sktime.transformations.series.fourier import FourierFeatures
''' example from sktime documentation to help understand implementation of FF
from sktime.datasets import load_airline
y_toy = load_airline()
transformer = FourierFeatures(sp_list=[12], fourier_terms_list=[4])
y_hat = transformer.fit_transform(y_toy)
display(y_toy.head(), y_hat.head())
#https://robjhyndman.com/hyndsight/forecasting-weekly-data/ #shows how to find best value of k
'''
ff_transformer = FourierFeatures(sp_list=[24,24*7,24*365.25], fourier_terms_list=[5,5,5]) #k=5 common value in energy forecasting
load_ff = ff_transformer.fit_transform(ts_load['total load actual'])
display(load_ff.head())

Now that I have the seasonalities represented as Fourier terms, I can use the SARIMAX model with autoARIMA for tuning to find the best values of pdq.

In [ ]:
#merge the new Fourier df with the features_min df (that excludes engineered features used in regression, 
#and just reverts back to only weather variables)
ff_combined = features_min.drop(['total load actual'], axis=1).merge(load_ff, left_index=True, right_index=True)
ff_combined['total load actual'] = features_min['total load actual']
ff_combined.head()

In [ ]:
#then create a train-test split with new Fourier terms:
#ts_train_test(data, target_col_name = 'total load actual', test_size=0.15, stdzd=False, cols_to_scale=None)
X_train_ff, X_test_ff, y_train, y_test = ts_train_test(data = ff_combined, stdzd=True, 
                                                 cols_to_scale=['temp','pressure','humidity',
                                                                'wind_speed','rain_1h','snow_3h','clouds_all'])
display(X_train_ff.head(3), X_test_ff.head(3), X_train_ff.columns)

I will try plotting the auto and partial autocorrelations to try to determine the q (moving average- MA) and p (auto regression -AR) terms of the ARIMA model. The daily trend is readily apparent in the ACF plot:

In [ ]:
#plot ACF and PACF
fig,ax = plt.subplots(2,1,figsize=(15,8))
fig = sm.graphics.tsa.plot_acf(y_train, lags=100, ax=ax[0])
fig = sm.graphics.tsa.plot_pacf(y_train, lags=100, ax=ax[1])
ax[0].set_xlim(0, 102)
ax[0].set_ylim(-0.4,1)
ax[1].set_xlim(0, 102)
plt.show()

Next I will apply first and 24-hour seasonal differences to see how they affect the stationarity of the data:

In [ ]:
ytrain_seas = y_train.diff(24)
ytrain_diff = y_train.diff()
ytrain_both = ytrain_seas.diff()
ytrain_both.tail()

In [ ]:
fig,ax = plt.subplots(2,1,figsize=(15,8))
fig = sm.graphics.tsa.plot_acf(ytrain_both[25:], lags=100, ax=ax[0])
fig = sm.graphics.tsa.plot_pacf(ytrain_both[25:], lags=100, ax=ax[1])
ax[0].set_xlim(-1, 105)
ax[0].set_ylim(-0.4,1)
ax[1].set_xlim(-1, 105)
ax[1].set_ylim(-0.4,1)
plt.show()

After differencing and seasonal differencing, the ACF and PACF plots still do not give much indication of which terms would be good for the AR and MA terms of the model.
So given the above information, a first pass at SARIMA parameters might be: (1,0,1)x(1,0,1,24) since d=0 for no trend.

In [ ]:
#%conda install -c conda-forge pmdarima
#%conda install -c conda-forge statsforecast
#import pmdarima as pm
#from sktime.forecasting.arima import AutoARIMA #https://alkaline-ml.com/pmdarima/tips_and_tricks.html
#https://medium.com/@josemarcialportilla/using-python-and-auto-arima-to-forecast-seasonal-time-series-90877adff03c

''' # It seems my computer cannot handle processing AutoARIMA for a dataset of this size. 
#Instead I try a few combinations of different orders individually below
stepwise_fit = pm.auto_arima(ytrain_both, X=X_train_ff,
                             start_p=0, d=0, start_q=0,                              
                             max_p=3, max_d=0, max_q=3,                       
                             start_P=0, D=0, start_Q=0,                             
                             max_P=3, max_D=0, max_Q=3,
                             m=24, seasonal=True,
                             information_criterion='aic',
                             stepwise=True,
                             error_action='ignore',
                             trace=True)
from sktime.forecasting.statsforecast import StatsForecastAutoARIMA
stepwise_fit = StatsForecastAutoARIMA(start_p=0, d=0, start_q=0,
                                     max_p=2, max_q=2,
                                     start_P=0, D=0, start_Q=0,
                                     max_P=3, max_D=0, max_Q=3,
                                     sp=24, seasonal=True,
                                     stationary=True,
                                     information_criterion='aic',
                                     stepwise=True, trend=False)
#stepwise_fit.fit(ytrain_both)
#print(stepwise_fit.get_test_params())

'''                                     

Since my computer does not seem to be able to handle the processing required for AutoARIMA, I try a few (pdq) combinations individually below.

In [ ]:

sar_mod = sm.tsa.statespace.SARIMAX(y_train, order=(1,0,1), seasonal_order=(1,0,1,24), freq='H') #result AIC=474868
#sar_mod = sm.tsa.statespace.SARIMAX(ytrain_both, order=(0,0,1), seasonal_order=(0,0,1,24), freq='H') #res AIC=476327
#sar_mod = sm.tsa.statespace.SARIMAX(ytrain_both, order=(2,0,1), seasonal_order=(1,0,1,24), freq='H') # 474833
#sar_mod = sm.tsa.statespace.SARIMAX(ytrain_both, order=(3,0,1), seasonal_order=(1,0,1,24), freq='H') #474737
#sar_mod = sm.tsa.statespace.SARIMAX(ytrain_both, order=(7,0,1), seasonal_order=(1,0,1,24), freq='H') #474481.147
#sar_mod = sm.tsa.statespace.SARIMAX(y_train, order=(1,0,1), seasonal_order=(1,0,1,24), freq='H') #475989
#sar_mod = sm.tsa.statespace.SARIMAX(y_train, order=(3,0,1), seasonal_order=(1,0,1,24), freq='H') #475842
#sar_mod = sm.tsa.statespace.SARIMAX(y_train, order=(7,0,1), seasonal_order=(1,0,1,24), freq='H') #475412.641
#sar_modx = sm.tsa.statespace.SARIMAX(y_train, order=(7,0,1), seasonal_order=(1,0,1,24), freq='H', exog=X_train_ff) #475421
#sar_mod = sm.tsa.statespace.SARIMAX(ytrain_both, order=(7,0,1), seasonal_order=(1,0,1,24), freq='H', exog=X_train_ff) #474552

In [ ]:
#sar_results = sar_mod.fit(method = 'powell')
#sar_results.summary()
sar_res = sar_mod.fit(method = 'powell')
sar_res.summary()

After trying several ARIMA orders, most have very similar AIC and BIC values. So it would make sense to use the simplest order that is performing as well as the others. Below I implement the (1,0,1)x(1,0,1,24) ARIMA model without the exogenous regressors.

In [ ]:
#sar_res.plot_diagnostics(figsize=(12,8))

In [ ]:
#prediction on last week of training set w/o exogenous vars
y_pred_sar = sar_res.get_prediction(start=X_train_ff.index[-24*7], end=X_train_ff.index[-1])

y_pred_sar = y_pred_sar.predicted_mean
y_pred_sar = pd.Series(y_pred_sar, index=X_train_ff.iloc[-24*7:,:].index)
error_metrics(y_true = y_train.iloc[-24*7:], y_pred = y_pred_sar, 
              model_name = "SARIMA (1,0,1)(1,0,1,24) prediction on last week of train set")

In [ ]:
#sar_modx = sm.tsa.statespace.SARIMAX(y_train, order=(1,0,1), seasonal_order=(1,0,1,24), freq='H', exog=X_train_ff)

In [ ]:
#sarx_results = sar_modx.fit(method = 'powell')
#sarx_results.summary()

In [ ]:
#prediction on last week of training set with exogenous
'''
y_pred_sarx = sarx_results.get_prediction(start=X_train_ff.index[-24*7], end=X_train_ff.index[-1])

y_pred_sarx = y_pred_sarx.predicted_mean
y_pred_sarx = pd.Series(y_pred_sarx, index=X_train_ff.iloc[-24*7:,:].index)
error_metrics(y_true = y_train.iloc[-24*7:], y_pred = y_pred_sarx, 
              model_name = "SARIMA (7,0,1)(1,0,1,24) prediction on last week of train set with exogenous regressors")
              '''

In [ ]:
#plot_ts_pred_true(y_pred = y_pred_sar, y_true = y_train.iloc[-24*7:], 
 #                 model_name = "SARIMA (7,0,1)(1,0,1,24) prediction on last week of train set without exogenous regressors")

In [ ]:
#prediction on first week of test set without exogenous regressors
y_pred2 = sar_res.get_forecast(steps=24*7)
#y_pred_ci = y_pred.conf_int()
y_pred2 = y_pred2.predicted_mean
y_pred2 = pd.Series(y_pred2, index=X_test_ff.iloc[:24*7,:].index)
error_metrics(y_true = y_test.iloc[:24*7], y_pred = y_pred2, 
              model_name = "SARIMA (1,0,1)(1,0,1,24) prediction on first week of test set")

In [ ]:
#prediction on first week of test set with exogenous
'''
y_pred3 = sarx_results.get_forecast(steps=24*7, exog=X_test_ff.iloc[:24*7,:])
#y_pred_ci = y_pred.conf_int()
y_pred3 = y_pred3.predicted_mean
y_pred3 = pd.Series(y_pred3, index=X_test_ff.iloc[:24*7,:].index)
error_metrics(y_true = y_test.iloc[:24*7], y_pred = y_pred3, 
              model_name = "SARIMA (1,0,1)(1,0,1,24) prediction on first week of test set with exogenous regressors")
              '''

In [ ]:
plot_ts_pred_true(y_pred = y_pred2, y_true = y_test.iloc[:24*7], 
                  model_name = "SARIMA (1,0,1)(1,0,1,24) prediction on first week of test set without exogenous regressors")

So the SARIMA(X) model performs somewhat poorly even just one week into the test set (1 week ahead of the end of the training set) both with and without the exogenous regressors, and slightly better without than with them. It catches the correct patterns but pretty consistently underpredicts. And as soon as the second seasonality kicks in at the end of the week (the model is handling only the daily seasonality), the accuracy drops significantly.

NOTE: after seeing that Prophet misses the mark badly on 2018-06-02 also, I did some digging and discovered that Pedro Sánchez was sworn in as Spain’s new prime minister that day, and so likely many more people than normal were indoors watching the swearing in on TV. See the discussion ater the Prophet plots for further insights.

Given this situation, let's see what the error is on just the first 6 days of the test set:

In [ ]:
#prediction on first 6 days of test set without exogenous regressors
y_pred2 = sar_res.get_forecast(steps=24*6)
#y_pred_ci = y_pred.conf_int()
y_pred2 = y_pred2.predicted_mean
y_pred2 = pd.Series(y_pred2, index=X_test_ff.iloc[:24*6,:].index)
error_metrics(y_true = y_test.iloc[:24*6], y_pred = y_pred2, 
              model_name = "SARIMA on first 6 days of test set")

That's quite a bit better without the one anomalous day and probably more in line with what we could expect from a short to medium term forecast with SARIMAX on average with this dataset.

## FB Prophet

Unlike SARIMAX, Prophet handles multi-seasonality well so I do not need to pass in the Fourier terms separately. The exogenous variables would return to just the weather variables. Prophet needs the time data in a specific format (not as index). And the target variable column needs to be named 'y'. You can add seasonalities, regressors, and even holidays in a given country.

In [ ]:
%pip install prophet 

In [ ]:
#%conda install -c conda-forge prophet
from prophet import Prophet

#for now, let's try with just the basic weather vars as our regressors
#prophet_min = combined_avg.iloc[:,16:].drop(['price day ahead', 'price actual','total load forecast'], axis=1)
#features_min = df_features.copy()
prophet_min = features_min.copy()
prophet_min.head(3)

In [ ]:
# function to transform data to format needed for Prophet
def prophetize(df):
    df_prophetized = df.reset_index().rename(columns={'total load actual':'y', 'time':'ds'})
    df_prophetized['ds'] = df_prophetized['ds'].dt.tz_localize(None)
    return df_prophetized

In [ ]:
# 'prophetize' the data:
prophet_df = prophetize(prophet_min)
prophet_df.head(3)

In [ ]:
#perform train-test split and Scaling
X_trainP, X_testP, y_trainP, y_testP = ts_train_test(data = prophet_df, target_col_name = 'y', stdzd=True, 
                                                 cols_to_scale=['temp','pressure','humidity',
                                                                'wind_speed','rain_1h','snow_3h','clouds_all'])

In [ ]:
# create train and test sets for Prophet
trainP = pd.merge(X_trainP, y_trainP, left_index=True, right_index=True)
testP = pd.merge(X_testP, y_testP, left_index=True, right_index=True)
trainP.head()

In [ ]:
#create instance of Prophet model
proph_mod = Prophet(interval_width = 0.95, 
                yearly_seasonality='auto',
                weekly_seasonality='auto',
                daily_seasonality='auto',
                seasonality_mode='additive')

In [ ]:
#add the regressor variables:
proph_mod.add_regressor('temp', standardize=False)
proph_mod.add_regressor('pressure', standardize=False)
proph_mod.add_regressor('humidity', standardize=False)
proph_mod.add_regressor('wind_speed', standardize=False)
proph_mod.add_regressor('rain_1h', standardize=False)
proph_mod.add_regressor('snow_3h', standardize=False)
proph_mod.add_regressor('clouds_all', standardize=False)

In [ ]:
# fit the model
proph_mod.fit(trainP)

In [ ]:
#dataframe with dt values to predict on (pred on train and test set)
future_dates = proph_mod.make_future_dataframe(periods=len(testP), freq='H', include_history=True)
# add regressors 
future_dates = pd.merge(future_dates, (pd.concat([trainP, testP])).drop('y', axis=1), on = 'ds')
future_dates.tail()

In [ ]:
# predict 
forecast = proph_mod.predict(future_dates)
forecast[['ds','yhat','yhat_lower','yhat_upper']].tail()

In [ ]:
#ploit using Prophet's built in plotting feature
_ = proph_mod.plot(forecast, uncertainty = True, xlabel = 'Time', ylabel = 'Total Load (MWH)')

The above plot was created with Prophet's built-in plot method. The black are the true values, dark blue the prediction, and light blue are the 95% confidence bands. It appears that the model does a reasonably good job overall, and predicts the lows quite well, while coming in a bit low on most of the peak loads.

In [ ]:
#plot the components of the results
_ = proph_mod.plot_components(forecast)

In [ ]:
#calculate error metrics on train data
error_metrics(y_true = trainP['y'], y_pred = forecast.iloc[:len(trainP)]['yhat'], 
              model_name = "FB Prophet Using Auto-Seasonality and Weather Regressors (train)")

In [ ]:
#error metrics on test data
error_metrics(y_true = testP['y'], y_pred = forecast.iloc[-len(testP):]['yhat'], 
              model_name = "FB Prophet Using Auto-Seasonality and Weather Regressors (test)")

So far, FB Prophet performs the best on the long range, multi-month forecast of all the models tested.

In [ ]:
#plot the entirety of the Prophet predicted and true loads
plot_ts_pred_true(y_pred = forecast.set_index('ds')['yhat'], y_true = pd.concat([trainP, testP]).set_index('ds')['y'], 
                  model_name = "FB Prophet Using Auto-Seasonality and Weather Regressors, train and test sets")

In [ ]:
#plot the test set only
plot_ts_pred_true(y_pred = forecast.set_index('ds')['yhat'].iloc[-len(testP):], 
                  y_true = pd.concat([trainP, testP]).set_index('ds')['y'].iloc[-len(testP):], 
                  model_name = "FB Prophet Using Auto-Seasonality and Weather Regressors, test set")

In [ ]:
#let's see how Prophet does on just the first week of the test set:
#dataframe with dt values to predict on (pred on train and test set)
future_dates_week = proph_mod.make_future_dataframe(periods=24*7, freq='H', include_history=True)
# add regressors 
future_dates_week = pd.merge(future_dates_week, (pd.concat([trainP, testP.iloc[:24*7]]).drop('y', axis=1)), on = 'ds')
future_dates_week.tail()

In [ ]:
#predict on the first week of test set
forecast_week = proph_mod.predict(future_dates_week)
forecast_week[['ds','yhat','yhat_lower','yhat_upper']].tail()

In [ ]:
#plot first week of test set
plot_ts_pred_true(y_pred = forecast_week.set_index('ds')['yhat'].iloc[-24*7:], 
                  y_true = pd.concat([trainP, testP.iloc[:24*7]]).set_index('ds')['y'].iloc[-24*7:], 
                  model_name = "FB Prophet Using Auto-Seasonality and Weather Regressors, first week of test set")

Here we again see the same as with SARIMAX, the model is underpredicting on the 7th day of the first week of the test set, which is a Saturday. Typically the weekends have a lower demand than weekdays, so I did some digging and the only remarkable event I could find was that Pedro Sánchez was sworn in as Spain’s new prime minister on Saturday June 2, 2018. So maybe the whole country was at home or at the bar, indoors at least, watching the swearing in on television and power demand was much higher than normal for a Saturday. This goes to show how a single event can throw off a model's prediction. It would be difficult for almost any model to handle something like this without a human manually telling the model to treat this day differently in some way. It looks like demand dips lower than normal on Friday too, maybe many people left work early and so demand dipped.

In [ ]:
#let's see the error metrics on just the first 6 days of the test set without the anomalous day:
error_metrics(y_true = testP['y'].iloc[:24*6], y_pred = forecast_week.iloc[-24*6:]['yhat'], 
              model_name = "FB Prophet, 6-day forecast (test)")

So we see that Prophet did better on a short-medium term forecast than it did on the long term forecast, and it performed similarly on the six day forecast compared to SARIMAX (SARIMAX had lower RMSE and R2 but higher MAPE).

Let's also see how well the Prophet model predicted compared to the TSO total load prediction (one of the columns of the original dataset; it is a 24-hr ahead forecast).

In [ ]:
#RMSE of actual load vs TSO forecast load
TSO_pred = combined_avg["total load forecast"].iloc[-len(testP):]
RMSE = np.sqrt(mean_squared_error(testP['y'], TSO_pred))
MAPE = (np.mean(np.abs((testP['y'] - TSO_pred) / testP['y'])) * 100)
print("RMSE= {0:.2f}".format(RMSE))

So on average for the whole test set, the RMSE between the TSO forecast and actual load was only about 341 MWH. But keep in mind that this is for a 24-hour ahead forecast, likely with the training set continuously updated to include the most recent data. 

## XGBoost

One other model to try would be Extreme Gradient Boosting (or XGBoost) which is very good at finding patterns in data and often provides better results than other ML algorithms. XGBoost can do a decent job with time series forecasting where there is seasonality but not much trend, as it cannot extrapolate. Since this dataset essentially has no trend, it may perform well.

In [ ]:
%pip install xgboost

In [ ]:
from xgboost import XGBRegressor

In [ ]:
#set up a parameter grid for hyperparameter Random CV search:

n_est = [int(n) for n in np.logspace(start=1, stop=2.5, num=10)]
max_depth = [0,3,6,9]
learn_rate = [0.1,0.2,0.3,0.4,0.5]

xgb_param_grid = {
        'n_estimators': n_est,
        'max_depth': max_depth,
        'learning_rate':learn_rate
}
xgb_param_grid

In [ ]:
#create XGB model instance
xgb_model = XGBRegressor()

#create TS splits for cross val (instead of random splits, this uses progressively larger sets starting from beginning)
tss = TimeSeriesSplit(n_splits=5)

#create instance of RandomSearchCV
xgb_cv = RandomizedSearchCV(xgb_model, param_distributions=xgb_param_grid, scoring='neg_root_mean_squared_error', cv=tss, random_state=47)

# Fit the random search model using reduced feature space X set
xgb_cv.fit(X_train, y_train)

xgb_cv.best_params_

In [ ]:
# calculate the RMSE of the model on the train set using the tuned hyperparameters
# best_params = {'n_estimators': 215, 'max_depth': 3, 'learning_rate': 0.4}
best_RMSE = xgb_cv.best_score_
print(-best_RMSE)

In [ ]:
# predict on the test set
xgb_pred = xgb_cv.predict(X_test)

In [ ]:
#calculate error metrics on train data
error_metrics(y_true = y_train, y_pred = xgb_cv.predict(X_train), 
              model_name = "XGBoost with weather data and engineered time-based features (train)")

In [ ]:
# plot the y_pred and y_true for train set
plot_ts_pred_true(y_pred = xgb_cv.predict(X_train), y_true = y_train, 
                  model_name = "XGBoost with weather data and engineered time-based features on train data")

In [ ]:
#calculate error metrics on test data
error_metrics(y_true = y_test, y_pred = xgb_pred, 
              model_name = "XGBoost with weather data and engineered time-based features (test)")

In [ ]:
# plot the y_pred and y_true for test set
plot_ts_pred_true(y_pred = xgb_pred, y_true = y_test, 
                  model_name = "XGBoost with weather data and engineered time-based features on test data")

# Conclusions
Multiple models and variations on them were used to forecast the total energy demand in Spain (in MWH) using hourly load and weather data. The demand is highly dependent on temperature as well as time of day and day of week. As a time series, it exhibits daily, weely, and yearly seasonalities, but does not display much of a trend. Below is a compilation of the various models and their error metrics on training and test data.

In [ ]:
#after running the error metric function on each model, the error_dict is populated w/ metrics for each
error_df = pd.DataFrame.from_dict(error_dict)

# create table and sort based on lowest RMSE
sorted_errors = np.round(error_df.pivot_table(index='Model Name', aggfunc='min').sort_values('RMSE', ascending=True),2)
sorted_errors

### Take Home: 
Based on RMSE, Prophet barely edges out XGBoost in terms of performance on the 7-month forecast of total energy demand, based on 41 months of training data. Prophet performs similarly to SARIMAX days into the forecast window, and with considerably lower training time. But since the SARIMA does so well on the training data, it may be useful on very short forecast windows, one to several hours into the future. Both have a place, as accurate long term forecasts are important for general planning and overall energy generation mix strategy, while hour-ahead (to several hours ahead) forecasts are critical for firing on fast-to-ramp-up systems for meeting peak demands.

### Further Investigation
Future efforts with this dataset could compare Prophet and/or XGBoost to the Spanish system operator's prediction of the total load. It would also be interesting to predict the contribution of wind generated energy given the wind speed and total load.